# 1、content的使用

举例1：保存多模态的数据，使用字典列表

In [1]:
import base64
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage
from dotenv import load_dotenv
import os

load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

def encode_image(img_path, img_type='jpeg'):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return f"data:image/{img_type};base64,{base64.b64encode(img_file.read()).decode("utf-8")}"

# 图像路径
img_path = "image_test.png"

# 获取图像base64编码字符串
base64_image = encode_image(img_path)

response = model.invoke(
    [
        HumanMessage(
            content=[
                {'type': 'text', 'text': '这张图里有什么？'},
                {
                    'type': 'image_url',
                    "image_url": base64_image,
                }
            ]
        )
    ]
)
print(response.content)

图中是一瓶香水，瓶身是透明的矩形玻璃，带金色瓶盖和金色装饰，摆在浅米色背景上，旁边有柔和的阴影。整体看起来像一张香水产品展示图。


# 2、content_blocks的使用

举例1 ：输入的格式化

In [4]:
from langchain.messages import HumanMessage
import os
from dotenv import load_dotenv

load_dotenv(override=True)

model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

def encode_image(img_path):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

# 图像路径
img_path = "image_test.png"

# 获取图像base64编码字符串
base64_image = encode_image(img_path)

response = model.invoke(
    [
        # 此种格式可用
        # HumanMessage(
        #     content=[
        #         {'type': 'text', 'text': '这张图里有什么？'},
        #         {
        #             'type': 'image_url',
        #             "image_url": base64_image,
        #         }
        #     ]
        # )
        # 推荐的统一写法
        HumanMessage(
            content_blocks=[
                {'type': 'text', 'text': '这张图里有什么？'},
                {
                    'type': 'image',
                    'base64': base64_image,
                    'mime_type': 'image/png',
                }
            ]
        )

    ]
)
print(response.content)

图里是一瓶香水。

它看起来是：
- 透明方形玻璃瓶身
- 金色瓶盖和装饰
- 浅金/香槟色调的包装风格
- 放在暖色背景上，带有柔和光影

整体给人的感觉比较精致、奢华。


作为对比

In [5]:
import base64
from langchain.messages import HumanMessage

from dotenv import load_dotenv
load_dotenv(override=True)

model = init_chat_model(
    model="claude-haiku-4-5",
    model_provider="openai",
    api_key=os.getenv("CLOSEAI_API_KEY"),
    base_url=os.getenv("CLOSEAI_BASE_URL")
)

def encode_image(img_path):
    """将一张本地图片转换成 Base64 编码的 Data URI 字符串,方便在文本中嵌入图片数据"""
    with open(img_path, "rb") as img_file:
        return base64.b64encode(img_file.read()).decode("utf-8")

# 图像路径
img_path = "image_test.png"

# 获取图像base64编码字符串
base64_image = encode_image(img_path)

response = model.invoke(
    [
        # 传统的写法：使用content。读取失败
        # HumanMessage(
        #     content=[
        #         {'type': 'text', 'text': '这张图里有什么？'},
        #         {
        #             'type': 'image_url',
        #             "image_url": base64_image,
        #         }
        #     ]
        # )

        # 推荐的统一写法
        HumanMessage(
            content_blocks=[
                {'type': 'text', 'text': '这张图里有什么？'},
                {
                    'type': 'image',
                    'base64': base64_image,
                    'mime_type': 'image/png',
                }
            ]
        )
    ]
)
print(response.content)

这张图里是一瓶**香水或液体化妆品**。具体特点包括：

- **瓶身**：透明玻璃瓶，呈方形或矩形设计
- **颜色**：液体呈浅金色或米色
- **瓶盖**：金色的喷雾泵头或压泵设计
- **风格**：高级、精致的美妆产品包装
- **背景**：米色/奶油色背景，光线充足

从外观推测，这可能是：
- 香水
- 粉底液
- 妆前乳或定妆喷雾
- 其他高端护肤/彩妆产品

整个画面呈现出专业的产品摄影风格，强调产品的质感和高级感。


举例2：输出的格式化

In [6]:
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

load_dotenv(override=True)

model = init_chat_model(
    model="deepseek:deepseek-v4-flash",
    extra_body={"thinking": {"type": "enabled"}},
)

response = model.invoke("你好，一句话回答")
print('=' * 20, '-> response <-', '=' * 20)
print(response)
print('=' * 20, '-> response.content <-', '=' * 20)
print(response.content)
print('=' * 20, '-> response.content_blocks <-', '=' * 20)
print(response.content_blocks)

==================== -> response <- ====================
content='你好，有什么我可以为你做的吗？' additional_kwargs={'refusal': None, 'reasoning_content': '嗯，用户要求“一句话回答”，说明需要简洁直接的回应。目前用户只是打招呼说“你好”，没有提出具体问题。根据常见对话开场白，用户可能只是礼貌问候，或者等待我先回应。考虑到“一句话回答”的指令，可以用简单友好的问候语回复，保持开放性，让用户知道我可以提供帮助。'} response_metadata={'token_usage': {'completion_tokens': 78, 'prompt_tokens': 8, 'total_tokens': 86, 'completion_tokens_details': {'accepted_prediction_tokens': None, 'audio_tokens': None, 'reasoning_tokens': 69, 'rejected_prediction_tokens': None}, 'prompt_tokens_details': {'audio_tokens': None, 'cached_tokens': 0}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 8}, 'model_provider': 'deepseek', 'model_name': 'deepseek-v4-flash', 'system_fingerprint': 'fp_8b330d02d0_prod0820_fp8_kvcache_20260402', 'id': '2f413c82-19af-4d07-aae7-7757867e5746', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e504f-4bc5-72a3-9762-9e3cc4a0bb46-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 8, 'ou